In [1]:
print("Kernel started successfully!")

Kernel started successfully!


In [2]:
import torch
import numpy as np
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight


print(torch.__version__) 

2.6.0+cu124


In [ ]:
#!pip uninstall torch torchvision torchaudio -y
#!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
#!pip install --upgrade torch torchvision torchaudio
#!pip install ipywidgets
#!pip install resampy

In [3]:
from imblearn.over_sampling import SMOTE

# ==========================================
# 1. LOAD AND PREPARE DATA (Fixes the NameError)
# ==========================================
PROJECT_PATH = "Thesis_Data"
LABELS_PATH = "videos_with_sentiment_labels.csv"

df = pd.read_csv(LABELS_PATH)
visual_dict = np.load(f"{PROJECT_PATH}/visual_features_clip.npy", allow_pickle=True).item()
audio_dict = np.load(f"{PROJECT_PATH}/audio_features_vggish.npy", allow_pickle=True).item()

X_visual, X_audio, y_labels = [], [], []

for index, row in df.iterrows():
    v_id = row['video_id']
    label = row['majority_sentiment']
    if v_id in visual_dict and v_id in audio_dict:
        X_visual.append(visual_dict[v_id])
        X_audio.append(audio_dict[v_id])
        y_labels.append(label)


X_visual = np.array(X_visual)
X_audio = np.array(X_audio)
y_labels = np.array(y_labels)

le = LabelEncoder()
y_encoded = le.fit_transform(y_labels)


# Concatena visual e audio prima di fare SMOTE
X_combined = np.hstack([X_visual, X_audio])
sm = SMOTE(random_state=42)
X_resampled, y_resampled = sm.fit_resample(X_combined, y_encoded)
X_visual = X_resampled[:, :X_visual.shape[1]]
X_audio  = X_resampled[:, X_visual.shape[1]:]
y_encoded = y_resampled


## Cross Attention in Intermediate Fusion

In [4]:

MAX_EPOCHS = 200
PATIENCE = 10 

class CrossModalAttention(nn.Module):
    """
    Il visual branch usa l'audio come contesto tramite cross-attention.
    - Query  = visual features  (il modello 'chiede' informazioni all'audio)
    - Key/Value = audio features (l'audio 'risponde' con il suo contenuto)
    Il risultato è una versione delle feature visive arricchita dal contesto audio.
    """
    def __init__(self, dim=128, NUM_HEADS=2, DROPOUT=0.1):
        super().__init__()
        assert dim % NUM_HEADS == 0, "dim deve essere divisibile per NUM_HEADS"
        
        self.attn = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=NUM_HEADS,
            dropout=DROPOUT,
            batch_first=True   # input shape: (batch, seq, dim) — più intuitivo
        )
        self.norm = nn.LayerNorm(dim)
        self.Dropout = nn.Dropout(DROPOUT)



    def forward(self, visual_feats, audio_feats):
        """
        Args:
            visual_feats: (B, dim) — feature visive dopo l'encoder
            audio_feats:  (B, dim) — feature audio dopo l'encoder
        Returns:
            (B, dim) — visual features aggiornate con contesto audio
        """
        # MultiheadAttention si aspetta (B, seq_len, dim)
        # I nostri vettori sono già aggregati nel tempo → seq_len = 1
        q = visual_feats.unsqueeze(1)   # (B, 1, dim)
        k = audio_feats.unsqueeze(1)    # (B, 1, dim)
        v = audio_feats.unsqueeze(1)    # (B, 1, dim)

        attended, _ = self.attn(q, k, v)  # (B, 1, dim)
        attended = attended.squeeze(1)     # (B, dim)

        # Connessione residuale + normalizzazione
        out = self.norm(visual_feats + self.Dropout(attended))
        return out  # (B, dim)

class FusionWithCrossAttention(nn.Module):
    def __init__(self, visual_dim=512, audio_dim=128, SHARED_DIM=128, num_classes=3, NUM_HEADS=2, DROPOUT=0.1):
        super().__init__()

        # Encoder separati: proiettano entrambe le modalità in SHARED_DIM 
        self.visual_encoder = nn.Sequential(
            nn.Linear(visual_dim, 256),
            nn.LayerNorm(256), nn.GELU(), nn.Dropout(DROPOUT),
            #nn.Linear(512, 256),               # ← layer aggiuntivo
            nn.LayerNorm(256), nn.GELU(), nn.Dropout(DROPOUT),
            nn.Linear(256, SHARED_DIM),
            nn.LayerNorm(SHARED_DIM), nn.GELU()
        )
        self.audio_encoder = nn.Sequential(
            nn.Linear(audio_dim, 256),
            nn.LayerNorm(256), nn.GELU(), nn.Dropout(DROPOUT),
            #nn.Linear(512, 256),               # ← layer aggiuntivo
            nn.LayerNorm(256), nn.GELU(), nn.Dropout(DROPOUT),
            nn.Linear(256, SHARED_DIM),
            nn.LayerNorm(SHARED_DIM), nn.GELU()
        )

        # Cross-attention: visual guarda l'audio
        self.cross_attn = CrossModalAttention(dim=SHARED_DIM , NUM_HEADS=NUM_HEADS)
        self.cross_attn_a = CrossModalAttention(dim=SHARED_DIM , NUM_HEADS=NUM_HEADS)

        # Classifier sulla concatenazione (visual_attended || audio_enc)
        self.classifier = nn.Sequential(
            nn.Linear(SHARED_DIM  * 2, 128),
            nn.GELU(), nn.Dropout(DROPOUT),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, visual_x, audio_x):
        v = self.visual_encoder(visual_x)   # (B, SHARED_DIM )
        a = self.audio_encoder(audio_x)     # (B, SHARED_DIM )

        # Visual arricchito dal contesto audio
        v_attended = self.cross_attn(v, a)  # (B, SHARED_DIM )
        a_attended = self.cross_attn_a(a, v)   # audio guarda visual

        combined = torch.cat([v_attended, a_attended], dim=1)

        # Fusione finale
        #combined = torch.cat([v_attended, a], dim=1)  # (B, SHARED_DIM  * 2)
        return self.classifier(combined)
    

# ==========================================
# SETUP TRAINING & EARLY STOPPING
# ==========================================
weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_encoded), y=y_encoded)
class_weights_tensor = torch.tensor(weights, dtype=torch.float32)

k_folds = 5
skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)

all_true_labels = []
all_predictions = []
fold_accuracies = []


In [5]:
import logging
import sys
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, 
    recall_score, classification_report
)

logger = logging.getLogger()
logger.setLevel(logging.INFO)

file_handler = logging.FileHandler("training_log.txt", mode="w")
file_handler.setLevel(logging.INFO)

console_handler = logging.StreamHandler(sys.stdout)
console_handler.setLevel(logging.INFO)

formatter = logging.Formatter("%(asctime)s - %(message)s")
file_handler.setFormatter(formatter)
console_handler.setFormatter(formatter)

logger.handlers = []
logger.addHandler(file_handler)
logger.addHandler(console_handler)

def log(msg):
    logger.info(msg)

results = []

SHARED_DIMS = [128, 256] # rappresenta la dimensione dello spazio latente condiviso in cui vengono proiettate le due modalità (visuale e audio) prima della fusione.
NUM_HEADS_LIST = [2, 4, 8] # rappresenta il numero di "teste" nell'attenzione multi-testa del modulo di cross-attention. Più teste permettono al modello di catturare diversi aspetti del contesto audio-visivo, ma aumentano anche la complessità computazionale.
DROPOUTS = [0.3]
LEARNING_RATES = [0.0003, 0.0001, 0.00005]

for SHARED_DIM in SHARED_DIMS:
    for NUM_HEADS in NUM_HEADS_LIST:
        for LEARNING_RATE in LEARNING_RATES:
            for DROPOUT in DROPOUTS:

                fold_accuracies = []
                fold_f1_macro = []
                fold_f1_weighted = []
                fold_precision_macro = []
                fold_recall_macro = []

                skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

                for fold, (train_idx, val_idx) in enumerate(skf.split(X_visual, y_encoded)):

                    X_v_train, X_v_val = X_visual[train_idx], X_visual[val_idx]
                    X_a_train, X_a_val = X_audio[train_idx], X_audio[val_idx]
                    y_train, y_val = y_encoded[train_idx], y_encoded[val_idx]

                    scaler_v = StandardScaler()
                    X_v_train_scaled = scaler_v.fit_transform(X_v_train)
                    X_v_val_scaled = scaler_v.transform(X_v_val)

                    scaler_a = StandardScaler()
                    X_a_train_scaled = scaler_a.fit_transform(X_a_train)
                    X_a_val_scaled = scaler_a.transform(X_a_val)

                    train_loader = DataLoader(TensorDataset(
                        torch.tensor(X_v_train_scaled, dtype=torch.float32),
                        torch.tensor(X_a_train_scaled, dtype=torch.float32),
                        torch.tensor(y_train, dtype=torch.long)
                    ), batch_size=32, shuffle=True)

                    val_loader = DataLoader(TensorDataset(
                        torch.tensor(X_v_val_scaled, dtype=torch.float32),
                        torch.tensor(X_a_val_scaled, dtype=torch.float32),
                        torch.tensor(y_val, dtype=torch.long)
                    ), batch_size=32, shuffle=False)

                    model = FusionWithCrossAttention(
                        visual_dim=X_visual.shape[1],
                        audio_dim=X_audio.shape[1],
                        SHARED_DIM=SHARED_DIM,
                        num_classes=len(le.classes_),
                        NUM_HEADS=NUM_HEADS,
                        DROPOUT=DROPOUT
                    )

                    class_weights_tensor = torch.tensor(
                        compute_class_weight(
                            class_weight='balanced',
                            classes=np.unique(y_encoded),
                            y=y_encoded
                        ),
                        dtype=torch.float32
                    )

                    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
                    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

                    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                        optimizer, mode='min', factor=0.5, patience=PATIENCE
                    )

                    best_val_loss = float('inf')
                    epochs_no_improve = 0

                    for epoch in range(MAX_EPOCHS):
                        model.train()

                        for inputs_v, inputs_a, labels in train_loader:
                            optimizer.zero_grad()
                            outputs = model(inputs_v, inputs_a)
                            loss = criterion(outputs, labels)
                            loss.backward()
                            optimizer.step()

                        model.eval()
                        val_loss = 0.0

                        with torch.no_grad():
                            for inputs_v, inputs_a, labels in val_loader:
                                outputs = model(inputs_v, inputs_a)
                                loss = criterion(outputs, labels)
                                val_loss += loss.item()

                        val_loss /= len(val_loader)
                        scheduler.step(val_loss)

                        if val_loss < best_val_loss:
                            best_val_loss = val_loss
                            epochs_no_improve = 0
                            best_model_state = model.state_dict()
                        else:
                            epochs_no_improve += 1

                        if epochs_no_improve >= PATIENCE:
                            break

                    model.load_state_dict(best_model_state)
                    model.eval()

                    preds, labels_list = [], []

                    with torch.no_grad():
                        for inputs_v, inputs_a, labels in val_loader:
                            outputs = model(inputs_v, inputs_a)
                            _, p = torch.max(outputs, 1)
                            preds.extend(p.numpy())
                            labels_list.extend(labels.numpy())

                    # ── METRICHE ──────────────────────────────────────────
                    acc        = accuracy_score(labels_list, preds)
                    f1_macro   = f1_score(labels_list, preds, average='macro',    zero_division=0)
                    f1_weighted= f1_score(labels_list, preds, average='weighted', zero_division=0)
                    prec_macro = precision_score(labels_list, preds, average='macro',    zero_division=0)
                    rec_macro  = recall_score(labels_list, preds,    average='macro',    zero_division=0)

                    fold_accuracies.append(acc)
                    fold_f1_macro.append(f1_macro)
                    fold_f1_weighted.append(f1_weighted)
                    fold_precision_macro.append(prec_macro)
                    fold_recall_macro.append(rec_macro)

                # ── MEDIE SUI FOLD ────────────────────────────────────────
                mean_acc        = np.mean(fold_accuracies)
                mean_f1_macro   = np.mean(fold_f1_macro)
                mean_f1_weighted= np.mean(fold_f1_weighted)
                mean_prec       = np.mean(fold_precision_macro)
                mean_rec        = np.mean(fold_recall_macro)

                results.append({
                    "SHARED_DIM":   SHARED_DIM,
                    "NUM_HEADS":    NUM_HEADS,
                    "DROPOUT":      DROPOUT,
                    "LR":           LEARNING_RATE,
                    "ACC":          mean_acc,
                    "F1_macro":     mean_f1_macro,
                    "F1_weighted":  mean_f1_weighted,
                    "Precision":    mean_prec,
                    "Recall":       mean_rec,
                })

                log(
                    f"Config: SHARED_DIM={SHARED_DIM}, NUM_HEADS={NUM_HEADS}, "
                    f"DROPOUT={DROPOUT}, LR={LEARNING_RATE} | "
                    f"ACC={mean_acc:.4f}  F1_macro={mean_f1_macro:.4f}  "
                    f"F1_w={mean_f1_weighted:.4f}  Prec={mean_prec:.4f}  Rec={mean_rec:.4f}"
                )
# ── RISULTATI FINALI ───────────────────────────────────────────────
best = max(results, key=lambda x: x["F1_macro"])
log(f"\nBEST CONFIG: {best}")

# Poi ri-addestra con quella config e stampa il report completo
log("\n" + classification_report(labels_list, preds, target_names=le.classes_, zero_division=0))

2026-04-29 11:06:46,961 - Config: SHARED_DIM=128, NUM_HEADS=2, DROPOUT=0.3, LR=0.0003 | ACC=0.8398  F1_macro=0.8379  F1_w=0.8380  Prec=0.8392  Rec=0.8398
2026-04-29 11:07:07,220 - Config: SHARED_DIM=128, NUM_HEADS=2, DROPOUT=0.3, LR=0.0001 | ACC=0.8463  F1_macro=0.8447  F1_w=0.8448  Prec=0.8466  Rec=0.8463
2026-04-29 11:07:35,253 - Config: SHARED_DIM=128, NUM_HEADS=2, DROPOUT=0.3, LR=5e-05 | ACC=0.8643  F1_macro=0.8632  F1_w=0.8633  Prec=0.8643  Rec=0.8644
2026-04-29 11:07:54,985 - Config: SHARED_DIM=128, NUM_HEADS=4, DROPOUT=0.3, LR=0.0003 | ACC=0.8437  F1_macro=0.8406  F1_w=0.8405  Prec=0.8489  Rec=0.8438
2026-04-29 11:08:19,624 - Config: SHARED_DIM=128, NUM_HEADS=4, DROPOUT=0.3, LR=0.0001 | ACC=0.8501  F1_macro=0.8489  F1_w=0.8490  Prec=0.8510  Rec=0.8501
2026-04-29 11:08:49,141 - Config: SHARED_DIM=128, NUM_HEADS=4, DROPOUT=0.3, LR=5e-05 | ACC=0.8333  F1_macro=0.8311  F1_w=0.8311  Prec=0.8348  Rec=0.8333
2026-04-29 11:09:07,730 - Config: SHARED_DIM=128, NUM_HEADS=8, DROPOUT=0.3, LR

In [9]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(labels_list, preds)
log(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

2026-04-29 11:16:53,851 - Accuracy: 0.8896 (88.96%)


In [ ]:
#max(results, key=lambda x: x['ACC'])
#sorted(results, key=lambda x: x["ACC"], reverse=True)[:5]